# 02 - Analyze
Reads the raw JSON from the Collect stage (same `RUN_ID` folder in the Lakehouse), evaluates every analyzer against the checklist, and merges to `findings.json`. No Azure/Fabric auth is needed - analyzers are offline.

## Parameters

In [ ]:
# Parameters (injected by the pipeline)
GITHUB_REPO_URL = "https://github.com/biro98/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
GITHUB_REF = ""  # pinned release ref from the pipeline (blank = GITHUB_BRANCH tip)
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
RUN_ID = "latest"

# Analysis thresholds — blank means "use the default from config/thresholds.yaml".
# A non-blank value is exported as the matching env var so the analyzers pick it up.
ARCH_MONOLITH_THRESHOLD = ""
ARCH_PIPELINE_STALE_DAYS = ""
PERF_STALE_DAYS = ""
PERF_LONG_REFRESH_HOURS = ""
PERF_FAIL_RATIO_THRESHOLD = ""
PERF_MODEL_SIZE_WARN_MB = ""
PERF_MODEL_SIZE_CRITICAL_MB = ""
PERF_REFRESH_OVERLAP_MIN = ""
PERF_THROTTLE_CRITICAL_PCT = ""
PERF_THROTTLE_WARN_PCT = ""
PERF_JOB_FAIL_RATIO = ""
PERF_JOB_LONG_HOURS = ""
GOV_SHARE_VOLUME_THRESHOLD = ""
SEC_BROAD_ACCESS_THRESHOLD = ""
SEC_GATEWAY_MIN_VERSION = ""
COST_SMALL_WORKSPACE_THRESHOLD = ""

## Clone framework + install dependencies

In [ ]:
import os, sys, shutil, subprocess
WORK_ROOT = "/tmp/fabric-arch-review-run"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL
_ref = (GITHUB_REF or "").strip() or GITHUB_BRANCH
def _git(args, **kw):
    r = subprocess.run(["git"] + args, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        out = ((r.stderr or "") + (r.stdout or "")).strip()
        raise RuntimeError("git " + " ".join(args) + " failed (rc=" + str(r.returncode) + "): " + out)
    return r

subprocess.run(["git", "config", "--global", "--add", "safe.directory", REPO_DIR])

def _fresh_clone():
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    _git(["clone", "--branch", _ref, "--depth", "1", _url, REPO_DIR])

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        _git(["-C", REPO_DIR, "remote", "set-url", "origin", _url])
        _git(["-C", REPO_DIR, "fetch", "--depth", "1", "origin", _ref])
        _git(["-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"])
    except RuntimeError:
        _fresh_clone()
else:
    _fresh_clone()
del _url
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")])
if _pip.returncode != 0:
    print("WARNING: pip install exited", _pip.returncode, "- a benign dependency-resolver warning is expected on Fabric; continuing.")
print("Repo + deps ready at", REPO_DIR)

## Analyze + merge findings

In [ ]:
import os, subprocess, sys
os.environ["CLIENT_NAME"] = CLIENT_NAME
os.environ["ENGAGEMENT_NAME"] = ENGAGEMENT_NAME
os.environ["REVIEWER_NAME"] = REVIEWER_NAME

# Export only the thresholds that were explicitly set in the Run dialog. A blank
# value is left unset so the analyzers fall back to config/thresholds.yaml (then
# their built-in default). The analyzers run as subprocesses below and inherit
# this environment.
for _k in (
    "ARCH_MONOLITH_THRESHOLD", "ARCH_PIPELINE_STALE_DAYS",
    "PERF_STALE_DAYS", "PERF_LONG_REFRESH_HOURS", "PERF_FAIL_RATIO_THRESHOLD",
    "PERF_MODEL_SIZE_WARN_MB", "PERF_MODEL_SIZE_CRITICAL_MB", "PERF_REFRESH_OVERLAP_MIN",
    "PERF_THROTTLE_CRITICAL_PCT", "PERF_THROTTLE_WARN_PCT",
    "PERF_JOB_FAIL_RATIO", "PERF_JOB_LONG_HOURS",
    "GOV_SHARE_VOLUME_THRESHOLD", "SEC_BROAD_ACCESS_THRESHOLD",
    "SEC_GATEWAY_MIN_VERSION", "COST_SMALL_WORKSPACE_THRESHOLD",
):
    _v = globals().get(_k, "")
    if str(_v).strip():
        os.environ[_k] = str(_v).strip()
        print("threshold override:", _k, "=", os.environ[_k])

LH = "/lakehouse/default/Files"
_BASE = os.path.join(LH, "fabric-arch-review")
if RUN_ID == "latest" and os.path.isdir(_BASE):
    _runs = [d for d in os.listdir(_BASE) if os.path.isdir(os.path.join(_BASE, d, "raw"))]
    if _runs:
        RUN_ID = max(_runs, key=lambda d: os.path.getmtime(os.path.join(_BASE, d)))
        print("Resolved RUN_ID=latest ->", RUN_ID)
OUT_DIR = os.path.join(LH, "fabric-arch-review", RUN_ID)
RAW_DIR = os.path.join(OUT_DIR, "raw")
if not os.path.isdir(RAW_DIR):
    raise RuntimeError("Raw folder not found: " + RAW_DIR + " -- run the Collect stage first.")
os.environ["OUTPUT_DIR"] = OUT_DIR

def _run(mod, args):
    print("\n==>", mod)
    r = subprocess.run([sys.executable, "-m", mod, *args], cwd=REPO_DIR)
    if r.returncode:
        print("  ", mod, "exit", r.returncode)

_run("analyzers.tenant_settings_review", ["--raw", os.path.join(RAW_DIR, "tenant_settings.json"),
                                          "--out", os.path.join(OUT_DIR, "findings_tenant_settings.json")])
for mod, fn in [
    ("analyzers.architecture_review", "findings_architecture.json"),
    ("analyzers.performance_review", "findings_performance.json"),
    ("analyzers.semantic_model_storage_review", "findings_storage_mode.json"),
    ("analyzers.governance_review", "findings_governance.json"),
    ("analyzers.security_review", "findings_security.json"),
    ("analyzers.cost_review", "findings_cost.json"),
    ("analyzers.notebook_code_review", "findings_notebook_code.json"),
    ("analyzers.best_practices_review", "findings_best_practices.json"),
]:
    _run(mod, ["--raw-dir", RAW_DIR, "--out", os.path.join(OUT_DIR, fn)])
_run("analyzers.merge_findings", ["--out-dir", OUT_DIR])
try:
    import notebookutils
    notebookutils.notebook.exit("analyzed:" + OUT_DIR)
except Exception:
    pass